In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit

In [0]:
dbutils.widgets.text("catalogo", "etl_supermercado")
dbutils.widgets.text("esquema",  "bronze")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema  = dbutils.widgets.get("esquema")

ruta = f"/Workspace/Users/fyamile.0428@gmail.com/etl-supermercado-databricks/datasets"

**ORDER PRODUCTS**

In [0]:
op_schema = StructType([StructField("order_id", IntegerType(), True),
                        StructField("product_id", IntegerType(), True),
                        StructField("add_to_cart_order", IntegerType(), True),
                        StructField("reordered", IntegerType(), True),
])

In [0]:
df_op = spark.read.option("header", True).option("sep", ";") \
    .schema(op_schema).csv(f"{ruta}/order_products__prior.csv") \
    .withColumn("ingestion_date", current_timestamp())

In [0]:
df_op.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.order_products")
print(f"✔ order_products: {df_op.count():,} filas")

**PRODUCTS**

In [0]:
prod_schema = StructType([StructField("product_id", IntegerType(), False),
                        StructField("product_name", StringType(), True),
                        StructField("aisle_id", IntegerType(), True),
                        StructField("department_id", IntegerType(), True)
])

In [0]:
df_prod = spark.read.option("header", True).option("sep", ";") \
    .schema(prod_schema).csv(f"{ruta}/products.csv") \
    .withColumn("ingestion_date", current_timestamp())

In [0]:
df_prod.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.products")
print(f"✔ products: {df_prod.count():,} filas")

**AISLES**

In [0]:
aisles_schema = StructType([StructField("aisle_id", IntegerType(), True),
                            StructField("aisle", StringType(), True),
])

In [0]:
df_aisles = spark.read.option("header", True).option("sep", ";") \
    .schema(aisles_schema).csv(f"{ruta}/aisles.csv") \
    .withColumn("ingestion_date", current_timestamp())

In [0]:
df_aisles.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.aisles")
print(f"✔ aisles: {df_aisles.count():,} filas")

**DEPARTMENTS**

In [0]:
dept_schema = StructType([StructField("department_id", IntegerType(), True),
                        StructField("department", StringType(),True),
])

In [0]:
df_dept = spark.read.option("header", True).option("sep", ";") \
    .schema(dept_schema).csv(f"{ruta}/departments.csv") \
    .withColumn("ingestion_date", current_timestamp())

In [0]:
df_dept.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.departments")
print(f"✔ departments: {df_dept.count():,} filas")